# 02 · Popularity-based recommender

Non-personalised baseline: every user gets the same list (minus movies they already rated). Three scoring rules:

* **count** — number of ratings $n_i$;
* **mean** — average rating $\bar r_i$;
* **bayesian** — damped mean $\tilde r_i = \dfrac{\sum_u r_{ui} + m\,\mu}{n_i + m}$, which shrinks the mean of rarely rated movies towards the global mean $\mu$ and so uses both the average and the number of ratings. $m$ acts as a prior of $m$ “virtual” ratings equal to $\mu$.

For rating prediction (RMSE) the model predicts $\tilde r_i$.

In [1]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)

def save_params(name, params):
    path = RESULTS / "best_params.json"
    allp = json.loads(path.read_text()) if path.exists() else {}
    allp[name] = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
    path.write_text(json.dumps(allp, indent=2))

In [2]:
from src.data import load_ratings, load_movies, temporal_user_split
ratings, movies = load_ratings(), load_movies()
train, test = temporal_user_split(ratings, test_frac=0.2)
train_in, valid = temporal_user_split(train, test_frac=0.2)
title = movies.set_index("item_idx")["title"]
print(f"train={len(train)} test={len(test)} | train_in={len(train_in)} valid={len(valid)}")

train=80000 test=20000 | train_in=64005 valid=15995


In [3]:
from src.models import PopularityRecommender
from src.evaluation import grid_search, evaluate_ranking, evaluate_rmse
tops = {}
for method in ["count", "mean", "bayesian"]:
    m = PopularityRecommender(method=method, m=20).fit(train)
    tops[method] = title[np.argsort(-m.item_scores_)[:10]].values
pd.DataFrame(tops, index=range(1, 11))

,count,mean,bayesian
1,Star Wars (1977),Everest (1998),Schindler's List (1993)
2,Fargo (1996),Someone Else's America (1995),Casablanca (1942)
3,Contact (1997),They Made Me a Criminal (1939),"Shawshank Redemption, The (1994)"
4,Return of the Jedi (1983),Aiqing wansui (1994),"Wrong Trousers, The (1993)"
5,"English Patient, The (1996)",Star Kid (1997),"Usual Suspects, The (1995)"
6,Liar Liar (1997),Prefontaine (1997),"Close Shave, A (1995)"
7,Scream (1996),Marlene Dietrich: Shadow and Light (1996),Star Wars (1977)
8,Toy Story (1995),Santa with Muscles (1996),Rear Window (1954)
9,Air Force One (1997),"Saint of Fort Washington, The (1993)","Silence of the Lambs, The (1991)"
10,Independence Day (ID4) (1996),Lamerica (1994),"Godfather, The (1972)"


The raw mean surfaces obscure movies rated 5 by one or two people; count and the damped mean give well-known titles.

## Hyper-parameter selection on the validation split

`m` only affects the `bayesian` rule, so the `count` and `mean` rows repeat.

In [4]:
grid_rank = grid_search(PopularityRecommender, {"method": ["count", "mean", "bayesian"], "m": [5, 20, 50, 100]},
                        train_in, valid, objective="ndcg@10", verbose=False)
grid_rank

,method,m,precision@10,recall@10,ndcg@10,coverage@10
0,count,5,0.0568,0.0815,0.0774,0.0386
1,count,20,0.0568,0.0815,0.0774,0.0386
2,count,50,0.0568,0.0815,0.0774,0.0386
3,count,100,0.0568,0.0815,0.0774,0.0386
4,bayesian,100,0.0518,0.0573,0.0686,0.0333
5,bayesian,50,0.0465,0.0492,0.0563,0.0333
6,bayesian,20,0.0431,0.0459,0.0486,0.0303
7,bayesian,5,0.0353,0.0393,0.0382,0.0268
8,mean,100,0.0002,0.0001,0.0004,0.0089
9,mean,50,0.0002,0.0001,0.0004,0.0089


In [5]:
grid_rmse = grid_search(lambda m: PopularityRecommender("bayesian", m), {"m": [0, 1, 5, 10, 20, 50, 100]},
                        train_in, valid, objective="rmse", verbose=False)
grid_rmse

,m,rmse
0,1,1.0383
1,5,1.0431
2,0,1.0455
3,10,1.0508
4,20,1.0617
5,50,1.0792
6,100,1.0930


In [6]:
best_rank = grid_rank.iloc[0][["method", "m"]].to_dict()
best_rmse = {"method": "bayesian", "m": grid_rmse.iloc[0]["m"]}
save_params("popularity_ranking", best_rank); save_params("popularity_rating", best_rmse)
grid_rank.to_csv(RESULTS / "grid_popularity_ranking.csv", index=False)
grid_rmse.to_csv(RESULTS / "grid_popularity_rmse.csv", index=False)
best_rank, best_rmse

({'method': 'count', 'm': 5}, {'method': 'bayesian', 'm': np.float64(1.0)})

## Test-set performance (selected configuration, refit on the full training set)

In [7]:
rank_model = PopularityRecommender(**best_rank).fit(train)
rate_model = PopularityRecommender(**best_rmse).fit(train)
res = evaluate_ranking(rank_model, train, test, ks=(10,))
res["rmse"] = evaluate_rmse(rate_model, test)
pd.Series(res)

precision@10      0.0726
recall@10         0.0722
ndcg@10           0.0909
coverage@10       0.0428
n_users         906.0000
rmse              1.0688
dtype: float64